# Ranga Production — Gemma 4 E2B (Google Colab → Android)

> Fine-tune on Drive data, export **`gemma-model.litertlm`**, deploy via `app/lib/services/gemma_service.dart`.

- [Gemma 4 LiteRT-LM](https://developers.google.com/edge/litert-lm/models/gemma-4)
- [Unsloth Gemma 4 guide](https://unsloth.ai/docs/models/gemma-4/train)

## How to run in Google Colab

1. In **Google Drive**, go to `My Drive/capstone/training/notebooks/`
2. Right-click this notebook → **Open with → Google Colaboratory**
3. **Runtime → Change runtime type → T4 GPU** (or L4/A100 for E2B production)
4. Left sidebar **🔑 Secrets** → add `HF_TOKEN` (your [Hugging Face token](https://huggingface.co/settings/tokens))
5. Run **all cells in order** from top to bottom
6. After the **Install** cell finishes → **Runtime → Restart session** → run from **Colab setup** onward

### Required Drive folders

```
My Drive/capstone/
├── training/          ← this notebook + src/ranga_train/
└── dataset/
    └── ranga_output/  ← ranga_sft_500.jsonl, ranga_eval_50.jsonl, ranga_tools.json
```


**GPU:** use **L4 or A100** if available (E2B LoRA needs ~10 GB VRAM).

In [ ]:
# @title 1. Install dependencies
# Run once, then: Runtime -> Restart session, and continue from "Colab setup".
!pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install -U torch tensorboard "transformers>=4.52" datasets accelerate trl protobuf sentencepiece matplotlib pandas ipywidgets --quiet
!pip install -U litert-torch-nightly --quiet

In [ ]:
# @title Colab setup — mount Drive, auth, GPU, paths
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from huggingface_hub import login

CAPSTONE_ROOT = "/content/drive/MyDrive/capstone"  # @param {type:"string"}

# 1) Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

CAPSTONE = Path(CAPSTONE_ROOT)
assert CAPSTONE.exists(), f"Drive folder not found: {CAPSTONE}"

# 2) Hugging Face token (Colab secret)
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

# 3) GPU check
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> GPU"
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

# 4) Import project code from Drive
TRAINING_DIR = CAPSTONE / "training"
sys.path.insert(0, str(TRAINING_DIR / "src"))

from ranga_train.colab_paths import require_dataset, resolve_project_paths
from ranga_train.data import load_jsonl, prepare_eval_records, summarize_dataset
from ranga_train.evaluate import evaluate_model
from ranga_train.inference import make_generate_fn

paths = resolve_project_paths(capstone_root=CAPSTONE)
require_dataset(paths)

TRAINING_DIR = paths["training_dir"]
DATASET_DIR = paths["dataset_dir"]
REPO_ROOT = paths["repo_root"]
REPORTS_DIR = TRAINING_DIR / "results" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(TRAINING_DIR)

sft_records = load_jsonl(DATASET_DIR / "ranga_sft_500.jsonl")
eval_records = prepare_eval_records(
    DATASET_DIR / "ranga_eval_50.jsonl",
    DATASET_DIR / "ranga_tools.json",
)
print("Capstone root:", REPO_ROOT)
print("Training dir:", TRAINING_DIR)
print("Dataset dir:", DATASET_DIR)
print("Dataset summary:", summarize_dataset(sft_records))
print("Held-out eval queries:", len(eval_records))

## Load Gemma 4 E2B + LoRA

In [ ]:
# @title 2. Config + dataset
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer
from ranga_train.gemma4 import (
    Gemma4ProductionConfig,
    flutter_integration_notes,
    litert_export_command,
    prepare_gemma4_sft_dataset,
)

g4_config = Gemma4ProductionConfig(
    dataset_dir=DATASET_DIR,
    checkpoint_dir=TRAINING_DIR / "results" / "gemma4_e2b_checkpoints",
    merged_dir=TRAINING_DIR / "results" / "gemma4_e2b_merged",
    export_dir=TRAINING_DIR / "results" / "gemma4_e2b_litertlm",
    reports_dir=REPORTS_DIR,
)
for p in (g4_config.checkpoint_dir, g4_config.merged_dir, g4_config.export_dir):
    p.mkdir(parents=True, exist_ok=True)

hf_datasets = prepare_gemma4_sft_dataset(
    g4_config.sft_path,
    train_split=g4_config.train_split,
    seed=g4_config.seed,
    max_hospitals=g4_config.max_hospitals_in_tool_payload,
)
print(f"Train={len(hf_datasets['train'])}  Val={len(hf_datasets['validation'])}")

SYSTEM_PROMPT = next(m["content"] for m in sft_records[0]["messages"] if m["role"] == "system")
PROMPT_ROLE = "system"

In [ ]:
# @title 3. Load base E2B (4-bit LoRA)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=g4_config.base_model,
    max_seq_length=g4_config.max_seq_length,
    dtype=None,
    load_in_4bit=True,
    token=hf_token,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=g4_config.lora_r,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=g4_config.lora_alpha,
    lora_dropout=g4_config.lora_dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=g4_config.seed,
)
print("LoRA ready on", g4_config.base_model)

## Baseline evaluation

In [ ]:
# @title 4. Baseline metrics
FastLanguageModel.for_inference(model)
baseline_generate = make_generate_fn(model, tokenizer)
baseline_report = evaluate_model(
    model_label="gemma4_e2b_baseline",
    eval_records=eval_records,
    system_prompt=SYSTEM_PROMPT,
    generate_fn=baseline_generate,
    prompt_role=PROMPT_ROLE,
)
pd.DataFrame([baseline_report.to_dict()])

## SFT training

In [ ]:
# @title 5. Train LoRA
FastLanguageModel.for_training(model)
sft_args = SFTConfig(
    output_dir=str(g4_config.checkpoint_dir),
    max_length=g4_config.max_seq_length,
    packing=False,
    num_train_epochs=g4_config.num_train_epochs,
    per_device_train_batch_size=g4_config.per_device_train_batch_size,
    gradient_accumulation_steps=g4_config.gradient_accumulation_steps,
    learning_rate=g4_config.learning_rate,
    warmup_ratio=g4_config.warmup_ratio,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    bf16=True,
    report_to="tensorboard",
    seed=g4_config.seed,
)
trainer = SFTTrainer(
    model=model, args=sft_args,
    train_dataset=hf_datasets["train"],
    eval_dataset=hf_datasets["validation"],
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model(str(g4_config.checkpoint_dir / "last_adapter"))

In [ ]:
# @title 6. Loss curve
log_history = trainer.state.log_history
train_loss = [x["loss"] for x in log_history if "loss" in x]
eval_loss = [x["eval_loss"] for x in log_history if "eval_loss" in x]
epoch_train = [x["epoch"] for x in log_history if "loss" in x]
epoch_eval = [x["epoch"] for x in log_history if "eval_loss" in x]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epoch_train, train_loss, label="Train")
ax.plot(epoch_eval, eval_loss, label="Validation")
ax.set_title("Gemma 4 E2B LoRA — Ranga SFT")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend(); ax.grid(True, alpha=0.3)
plt.savefig(REPORTS_DIR / "gemma4_e2b_loss_curve.png", dpi=150)
plt.show()

## Merge, evaluate, export for phone

In [ ]:
# @title 7. Merge LoRA (required for LiteRT-LM)
merged_model = model.merge_and_unload()
merged_model.save_pretrained(g4_config.merged_dir)
tokenizer.save_pretrained(g4_config.merged_dir)
print("Merged weights:", g4_config.merged_dir)

In [ ]:
# @title 8. Post-SFT metrics + comparison
FastLanguageModel.for_inference(merged_model)
finetuned_generate = make_generate_fn(merged_model, tokenizer)
finetuned_report = evaluate_model(
    model_label="gemma4_e2b_finetuned",
    eval_records=eval_records,
    system_prompt=SYSTEM_PROMPT,
    generate_fn=finetuned_generate,
    prompt_role=PROMPT_ROLE,
)
comparison_df = pd.DataFrame(finetuned_report.comparison_table(baseline_report))
comparison_df

In [ ]:
# @title 9. Export .litertlm to Drive
export_cmd = litert_export_command(g4_config.merged_dir, g4_config.export_dir)
print(export_cmd)
!{export_cmd}

litert_files = list(g4_config.export_dir.glob("*.litertlm"))
assert litert_files, "Export failed — no .litertlm found."
bundle = litert_files[0]
print(f"Bundle: {bundle} ({bundle.stat().st_size / 1e9:.2f} GB)")
print(flutter_integration_notes(bundle))

In [ ]:
# @title 10. Save reports to Drive
(REPORTS_DIR / "gemma4_e2b_baseline.json").write_text(json.dumps(baseline_report.to_dict(), indent=2))
(REPORTS_DIR / "gemma4_e2b_finetuned.json").write_text(json.dumps(finetuned_report.to_dict(), indent=2))
comparison_df.to_csv(REPORTS_DIR / "gemma4_e2b_comparison.csv", index=False)
print("Reports:", REPORTS_DIR)
print("Upload", bundle.name, "to Hugging Face, then update gemma_service.dart _modelUrl")